# LUẬN VĂN TỐT NGHIỆP: PHÂN LOẠI 6 THỂ LOẠI NHẠC CỔ TRUYỀN VIỆT NAM
## Phase 5 (Group Split v2): CNN Transfer Learning Benchmark trên Spectrogram (Log-Mel & CQT)
## Phase 6 (Group Split v2): Multi-Modal Late Fusion (Best CNN + Fuzzy Random Forest / Raw Features)

Notebook này được tối ưu chạy trực tiếp trên **Google Colab (GPU T4 miễn phí)** theo **Protocol B (Song-Level Group Split)**:
- **Loại bỏ triệt để rò rỉ dữ liệu (0% Data Leakage):** Toàn bộ phân đoạn của cùng một bài hát (`audio_path`) chỉ được phép nằm trong 1 tập duy nhất.
- **Phân chia 3 tập chuẩn hóa:** 70% Train (2,520 mẫu) / 10% Validation (360 mẫu) / 20% Test (720 mẫu).
- **Early Stopping & Model Selection:** Checkpoint tốt nhất và cơ chế dừng sớm được quyết định bằng tập **Validation độc lập**, bảo vệ tập Test khỏi hiện tượng Data Snooping.
- **Phạm vi:** Huấn luyện lại và đối sánh 13 thực nghiệm CNN (EXP-01 đến EXP-13) và tự động Late Fusion ở Phase 6.
- **Tự động lưu & phục hồi:** Kết quả, checkpoint (.pth), ma trận nhầm lẫn (.png) và xác suất (.npy) được tự động đồng bộ tức thì vào Google Drive.


In [ ]:
# 1. KIỂM TRA PHẦN CỨNG GPU
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Menu: Runtime -> Change runtime type -> Chọn T4 GPU.")
!nvidia-smi


### 2. TẢI DỮ LIỆU SPECTROGRAMS VÀ FILE SPLIT GROUP V2 LÊN COLAB
Bạn cần chuẩn bị trên Google Drive thư mục `/content/drive/MyDrive/Colab Notebooks/LuanVan/` gồm:
1. `spectrograms_for_colab.zip` (chứa 3,600 ảnh Mel và 3,600 ảnh CQT).
2. File `split_group_v2.csv` (định nghĩa chính xác phân bổ Train / Val / Test theo từng bài hát).
Chạy ô dưới để tự động mount Drive, giải nén và kiểm tra dữ liệu.


In [ ]:
# 2. GIẢI NÉN DỮ LIỆU & KIỂM TRA FILE SPLIT GROUP V2
import os
import shutil
from pathlib import Path

# Mount Google Drive (nếu chưa mount)
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LuanVan')
zip_path = DRIVE_DIR / 'spectrograms_for_colab.zip'
split_path_drive = DRIVE_DIR / 'split_group_v2.csv'

mel_dir = Path('spectrograms/mel')
cqt_dir = Path('spectrograms/cqt')
mel_count = len(list(mel_dir.rglob('*.png'))) if mel_dir.exists() else 0
cqt_count = len(list(cqt_dir.rglob('*.png'))) if cqt_dir.exists() else 0

if mel_count != 3600 or cqt_count != 3600:
    if zip_path.exists():
        print(f'Phát hiện dữ liệu ảnh chưa đầy đủ (Mel: {mel_count}, CQT: {cqt_count}). Đang giải nén...')
        !rm -rf spectrograms
        !unzip -q -o "$zip_path" -d /content/
        print('[OK] Đã giải nén ảnh spectrograms thành công!')
    else:
        print(f'[LỖI] Không tìm thấy file zip tại: {zip_path}')
else:
    print('[OK] Đã có đầy đủ 3,600 ảnh Mel và 3,600 ảnh CQT!')

# Copy file split_group_v2.csv vào thư mục làm việc
local_split_path = Path('spectrograms/split_group_v2.csv')
if split_path_drive.exists():
    local_split_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(split_path_drive, local_split_path)
    print(f'[OK] Đã nạp thành công file split: {local_split_path}')
elif local_split_path.exists():
    print(f'[OK] Đã có sẵn file split cục bộ: {local_split_path}')
else:
    print('[CẢNH BÁO] Không tìm thấy split_group_v2.csv trên Drive. Bạn hãy upload file này lên thư mục Drive hoặc trực tiếp lên Colab!')


In [ ]:
# 3. KHỞI TẠO CÁC THƯ VIỆN & CẤU HÌNH PROTOCOL B (SONG-LEVEL GROUP SPLIT)
import os
import time
import math
import shutil
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Thiết lập Random Seed đảm bảo tính tái lập 100%
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1

# Tạo các thư mục lưu trữ cục bộ cho Protocol B (không ghi đè kết quả cũ của Protocol A)
Path('results_groupv2/cnn_confusion_matrices').mkdir(parents=True, exist_ok=True)
Path('models/cnn_groupv2').mkdir(parents=True, exist_ok=True)

# Thiết lập lưu vào Google Drive
DRIVE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LuanVan')
if DRIVE_DIR.exists():
    print(f'[KẾT NỐI DRIVE THÀNH CÔNG] Dữ liệu sẽ tự động lưu vào: {DRIVE_DIR}')
    (DRIVE_DIR / 'results_groupv2/cnn_confusion_matrices').mkdir(parents=True, exist_ok=True)
    (DRIVE_DIR / 'models/cnn_groupv2').mkdir(parents=True, exist_ok=True)
else:
    print('[CẢNH BÁO] Chưa kết nối Drive. Dữ liệu sẽ lưu tạm tại /content/results_groupv2')

GENRES = ['Ca trù', 'Chèo', 'Chầu văn', 'Cải lương', 'Hát xẩm', 'Quan họ']
print('Danh sách 6 thể loại chuẩn:', GENRES)


In [ ]:
# 4. ĐỊNH NGHĨA DATASET, SPECAUGMENT & LOAD DỮ LIỆU PROTOCOL B (TRAIN / VAL / TEST)
class SpecAugment(nn.Module):
    """SpecAugment: Che ngẫu nhiên dải thời gian và dải tần số trên ảnh phổ."""
    def __init__(self, time_mask_param=30, freq_mask_param=16):
        super().__init__()
        self.time_mask_param = time_mask_param
        self.freq_mask_param = freq_mask_param

    def forward(self, x):
        if not self.training:
            return x
        cloned = x.clone()
        B, C, H, W = cloned.shape
        f = random.randint(0, self.freq_mask_param)
        f0 = random.randint(0, H - f)
        cloned[:, :, f0:f0+f, :] = 0.0
        t = random.randint(0, self.time_mask_param)
        t0 = random.randint(0, W - t)
        cloned[:, :, :, t0:t0+t] = 0.0
        return cloned

class SpectrogramDataset(Dataset):
    def __init__(self, samples: list, transform=None, dual_channel=False):
        self.samples = samples
        self.transform = transform
        self.dual_channel = dual_channel

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p1, p2, label = self.samples[idx]
        if self.dual_channel:
            mel_img = Image.open(p1).convert('L')
            cqt_img = Image.open(p2).convert('L')
            mel_t = transforms.ToTensor()(mel_img)
            cqt_t = transforms.ToTensor()(cqt_img)
            img_t = torch.cat([mel_t, cqt_t], dim=0) # (2, H, W)
            img_t = transforms.Resize((IMG_SIZE, IMG_SIZE))(img_t)
            img_t = transforms.Normalize(mean=[0.5, 0.5], std=[0.5, 0.5])(img_t)
        else:
            img = Image.open(p1).convert('RGB')
            if self.transform:
                img_t = self.transform(img)
            else:
                img_t = transforms.ToTensor()(img)
        return img_t, label

def load_data_split(spec_type='mel'):
    """
    Nạp tập dữ liệu theo Protocol B (split_group_v2.csv):
    - Tách theo bài hát (audio_path), đảm bảo 0% rò rỉ giữa các tập.
    - Trả về 3 tập mẫu: train_samples (2,520), val_samples (360), test_samples (720).
    """
    split_file = Path('spectrograms/split_group_v2.csv')
    if not split_file.exists():
        split_file = Path('split_group_v2.csv')
    if not split_file.exists():
        split_file = DRIVE_DIR / 'split_group_v2.csv'
    
    df_meta = pd.read_csv(split_file)
    genre_to_idx = {g: i for i, g in enumerate(GENRES)}

    train_samples, val_samples, test_samples = [], [], []
    for _, row in df_meta.iterrows():
        label_idx = genre_to_idx[row['label']]
        p_mel = Path('spectrograms') / row['mel_rel_path']
        p_cqt = Path('spectrograms') / row['cqt_rel_path']

        if spec_type == 'mel':
            item = (p_mel, p_mel, label_idx)
        elif spec_type == 'cqt':
            item = (p_cqt, p_cqt, label_idx)
        else: # mel+cqt
            item = (p_mel, p_cqt, label_idx)

        sp = row['split']
        if sp == 'train':
            train_samples.append(item)
        elif sp == 'val':
            val_samples.append(item)
        elif sp == 'test':
            test_samples.append(item)

    print(f"[Data Split Protocol B] Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")
    return train_samples, val_samples, test_samples

def get_transforms(augment=True):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if augment:
        tr = transforms.Compose([
            transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
        ])
    else:
        tr = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    val = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return tr, val


In [ ]:
# 5. KHỞI TẠO CÁC KIẾN TRÚC MÔ HÌNH CNN TRANSFER LEARNING
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_efficientnet_b0(strategy='partial', in_channels=3):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    if in_channels == 2:
        old_conv = m.features[0][0]
        new_conv = nn.Conv2d(2, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride, padding=old_conv.padding, bias=False)
        with torch.no_grad():
            new_conv.weight.copy_(old_conv.weight[:, :2, :, :])
        m.features[0][0] = new_conv
    # Classifier Head (KHÔNG dùng inplace=True để tránh lỗi autograd modified inplace)
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.classifier[1].in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES)
    )
    return m

def build_efficientnet_b2(strategy='partial'):
    m = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.classifier[1].in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES)
    )
    return m

def build_mobilenet_v2(strategy='partial'):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    )
    return m

def build_resnet50(strategy='partial'):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES)
    )
    return m

def build_vgg16(strategy='partial'):
    m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    m.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, NUM_CLASSES)
    )
    return m


In [ ]:
# 6. ENGINE HUẤN LUYỆN & ĐÁNH GIÁ (VAL EARLY STOPPING, TEST EVALUATION 1 LẦN CUỐI)
def train_one_epoch(model, loader, optimizer, criterion, spec_aug, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs = spec_aug(imgs)
        optimizer.zero_grad()
        if scaler is not None:
            autocast_ctx = torch.amp.autocast('cuda') if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast()
            with autocast_ctx:
                out = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (out.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, (correct / total) * 100

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        out = model(imgs)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds, average='macro') * 100
    return acc, f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)

def run_experiment(exp_id, model_name, spec_type, strategy,
                   p1_lr=1e-3, p1_ep=4, p2_lr=1e-5, p2_ep=8):
    exp_name = f"EXP-{exp_id:02d}_{model_name}_{spec_type}_{strategy}"
    print(f"\n{'='*70}\n[{exp_name}] PROTOCOL B (SONG-LEVEL GROUP SPLIT)\n{'='*70}")

    dual = (spec_type == 'mel+cqt')
    train_samples, val_samples, test_samples = load_data_split(spec_type=spec_type)
    train_tf, val_tf = get_transforms(augment=True)

    train_ds = SpectrogramDataset(train_samples, transform=train_tf if not dual else None, dual_channel=dual)
    val_ds   = SpectrogramDataset(val_samples,   transform=val_tf   if not dual else None, dual_channel=dual)
    test_ds  = SpectrogramDataset(test_samples,  transform=val_tf   if not dual else None, dual_channel=dual)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

    in_ch = 2 if dual else 3
    if model_name == 'efficientnet_b0': model = build_efficientnet_b0(strategy, in_channels=in_ch)
    elif model_name == 'efficientnet_b2': model = build_efficientnet_b2(strategy)
    elif model_name == 'mobilenet_v2': model = build_mobilenet_v2(strategy)
    elif model_name == 'resnet50': model = build_resnet50(strategy)
    elif model_name == 'vgg16': model = build_vgg16(strategy)
    else: raise ValueError(f"Unknown model: {model_name}")

    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    spec_aug = SpecAugment(time_mask_param=30, freq_mask_param=16).to(DEVICE)

    if DEVICE.type == 'cuda':
        scaler = torch.amp.GradScaler('cuda') if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler()
    else:
        scaler = None

    best_model_path = Path(f'models/cnn_groupv2/{exp_name}_best.pth')
    val_probs_path = Path(f'models/cnn_groupv2/exp{exp_id:02d}_val_probs.npy')
    test_probs_path = Path(f'models/cnn_groupv2/exp{exp_id:02d}_test_probs.npy')
    best_val_macro_f1 = 0.0
    start_time = time.time()

    # 1. Đóng băng backbone
    for p in model.parameters():
        p.requires_grad = False

    # 2. Mở khóa head phân loại
    if model_name in ('efficientnet_b0', 'efficientnet_b2', 'mobilenet_v2'):
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif model_name == 'resnet50':
        for p in model.fc.parameters():
            p.requires_grad = True
    elif model_name == 'vgg16':
        for p in model.classifier.parameters():
            p.requires_grad = True

    print(f"  [Khởi tạo] Tham số trainable Phase 1: {count_trainable(model):,}")

    # Phase 1: Warmup Classifier Head (Đánh giá trên VAL SET)
    if p1_ep > 0:
        print(f"  [Phase 1] {p1_ep} epochs, LR={p1_lr}")
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=p1_lr, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=p1_ep, eta_min=1e-6)

        for ep in range(1, p1_ep + 1):
            t_ep = time.time()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, spec_aug, scaler)
            val_acc, val_f1, _, _, _ = evaluate(model, val_loader)
            scheduler.step()
            print(f"    Ep {ep:2d}/{p1_ep} ({time.time()-t_ep:.1f}s): loss={tr_loss:.4f} "
                  f"tr_acc={tr_acc:.2f}% | VAL_acc={val_acc:.2f}% VAL_f1={val_f1:.2f}%")
            if val_f1 > best_val_macro_f1:
                best_val_macro_f1 = val_f1
                torch.save(model.state_dict(), best_model_path)

    # Phase 2: Fine-tune Backbone (Early stopping dựa trên VAL SET)
    if p2_ep > 0 and strategy in ('partial', 'full'):
        if best_model_path.exists():
            model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

        if strategy == 'partial':
            if model_name == 'efficientnet_b0':
                for p in model.features[5:].parameters(): p.requires_grad = True
            elif model_name == 'efficientnet_b2':
                for p in model.features[4:].parameters(): p.requires_grad = True
            elif model_name == 'mobilenet_v2':
                for p in model.features[14:].parameters(): p.requires_grad = True
            elif model_name == 'resnet50':
                for p in model.layer4.parameters(): p.requires_grad = True
            elif model_name == 'vgg16':
                for p in model.features[24:].parameters(): p.requires_grad = True
        elif strategy == 'full':
            for p in model.parameters(): p.requires_grad = True

        print(f"  [Phase 2] {p2_ep} epochs, LR={p2_lr} | Tham số trainable: {count_trainable(model):,}")
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=p2_lr, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=p2_ep, eta_min=1e-7)
        patience = 0

        for ep in range(1, p2_ep + 1):
            t_ep = time.time()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, spec_aug, scaler)
            val_acc, val_f1, _, _, _ = evaluate(model, val_loader)
            scheduler.step()
            print(f"    Ep {ep:2d}/{p2_ep} ({time.time()-t_ep:.1f}s): loss={tr_loss:.4f} "
                  f"tr_acc={tr_acc:.2f}% | VAL_acc={val_acc:.2f}% VAL_f1={val_f1:.2f}%")
            if val_f1 > best_val_macro_f1:
                best_val_macro_f1 = val_f1
                torch.save(model.state_dict(), best_model_path)
                patience = 0
            else:
                patience += 1
                if patience >= 10:
                    print("    Early stopping trên tập Validation sau 10 epochs không cải thiện.")
                    break

    elapsed = (time.time() - start_time) / 60.0
    # Nạp lại checkpoint tốt nhất trên tập Validation
    if best_model_path.exists():
        model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    
    # Đánh giá 1 lần duy nhất trên tập VAL và TEST
    val_acc, val_f1, _, _, val_probs = evaluate(model, val_loader)
    test_acc, test_f1, preds, labels, test_probs = evaluate(model, test_loader)
    
    print(f"\n  --> KẾT QUẢ PROTOCOL B: Val Acc={val_acc:.2f}% Val F1={val_f1:.2f}% | TEST ACC={test_acc:.2f}%  TEST F1={test_f1:.2f}%  ({elapsed:.1f} phút)")

    np.save(val_probs_path, val_probs)
    np.save(test_probs_path, test_probs)
    
    report = classification_report(labels, preds, target_names=GENRES, output_dict=True)
    per_class_f1 = {g: round(report[g]['f1-score'] * 100, 2) for g in GENRES}

    # Lưu Ma trận nhầm lẫn Test Set
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=GENRES, yticklabels=GENRES)
    plt.title(f"{exp_name} (Protocol B - Group Split)\nTest Acc={test_acc:.2f}% | F1={test_f1:.2f}%")
    plt.xlabel("Dự đoán")
    plt.ylabel("Thực tế")
    plt.tight_layout()
    plt.savefig(f"results_groupv2/cnn_confusion_matrices/{exp_name}_cm.png", dpi=150)
    plt.close()

    res = {
        'exp_id': exp_id,
        'exp_name': exp_name,
        'model_name': model_name,
        'spec_type': spec_type,
        'strategy': strategy,
        'val_accuracy': round(val_acc, 4),
        'val_macro_f1': round(val_f1, 4),
        'test_accuracy': round(test_acc, 4),
        'macro_f1': round(test_f1, 4),
        'delta_vs_baseline': round(test_acc - 65.56, 4),
        'training_time_min': round(elapsed, 2),
        'trainable_params': count_trainable(model),
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M')
    }
    res.update({f'f1_{g.replace(" ","_")}': v for g, v in per_class_f1.items()})

    # Đồng bộ sang Google Drive
    if DRIVE_DIR.exists():
        try:
            if best_model_path.exists():
                shutil.copy2(best_model_path, DRIVE_DIR / 'models/cnn_groupv2/' / best_model_path.name)
            if val_probs_path.exists():
                shutil.copy2(val_probs_path, DRIVE_DIR / 'models/cnn_groupv2/' / val_probs_path.name)
            if test_probs_path.exists():
                shutil.copy2(test_probs_path, DRIVE_DIR / 'models/cnn_groupv2/' / test_probs_path.name)
            cm_p = Path(f'results_groupv2/cnn_confusion_matrices/{exp_name}_cm.png')
            if cm_p.exists():
                shutil.copy2(cm_p, DRIVE_DIR / 'results_groupv2/cnn_confusion_matrices/' / cm_p.name)
            print(f'  [DRIVE SYNC] Đã đồng bộ an toàn {exp_name} vào Google Drive!')
        except Exception as e:
            print(f'  [CẢNH BÁO DRIVE] Lỗi sao chép sang Drive: {e}')
    return res


In [ ]:
# 7. THỰC THI 13 THỰC NGHIỆM PROTOCOL B (TỰ ĐỘNG LƯU VÀO GOOGLE DRIVE)
# Cơ chế tự động nhận diện mô hình đã xong và chỉ chạy tiếp các mô hình còn lại!
EXPERIMENTS = [
    # id  model_name          spec_type   strategy   p1_lr  p1_ep  p2_lr  p2_ep
    ( 1, 'efficientnet_b0',  'mel',      'frozen',  1e-3,  25,    0,     0    ),
    ( 2, 'efficientnet_b0',  'mel',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 3, 'efficientnet_b0',  'mel',      'full',    1e-3,  6,     5e-6,  30   ),
    ( 4, 'efficientnet_b0',  'cqt',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 5, 'mobilenet_v2',     'mel',      'frozen',  1e-3,  25,    0,     0    ),
    ( 6, 'mobilenet_v2',     'mel',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 7, 'mobilenet_v2',     'cqt',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 8, 'resnet50',         'mel',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 9, 'resnet50',         'cqt',      'partial', 1e-3,  8,     1e-5,  25   ),
    (10, 'vgg16',            'mel',      'frozen',  1e-3,  20,    0,     0    ),
    (11, 'vgg16',            'mel',      'partial', 1e-3,  6,     1e-5,  20   ),
    (12, 'efficientnet_b2',  'mel',      'partial', 1e-3,  8,     1e-5,  25   ),
    (13, 'efficientnet_b0',  'mel+cqt',  'partial', 1e-3,  8,     1e-5,  25   ),
]

res_csv_path = Path('results_groupv2/cnn_benchmark_groupv2.csv')
drive_csv_path = DRIVE_DIR / 'results_groupv2/cnn_benchmark_groupv2.csv'

if drive_csv_path.exists():
    shutil.copy2(drive_csv_path, res_csv_path)
    print(f'[ĐÃ NẠP TỪ GOOGLE DRIVE] Tìm thấy file cnn_benchmark_groupv2.csv cũ trên Drive!')

if res_csv_path.exists():
    df_existing = pd.read_csv(res_csv_path)
    done_ids = set(df_existing['exp_id'].tolist())
    all_results = df_existing.to_dict('records')
    print(f'--> Danh sách các thực nghiệm ĐÃ HOÀN THÀNH: {sorted(list(done_ids))}')
else:
    done_ids = set()
    all_results = []

for cfg in EXPERIMENTS:
    eid = cfg[0]
    if eid in done_ids:
        print(f'--> [EXP-{eid:02d}] Đã hoàn thành trước đó -> BỎ QUA.')
        continue
    r = run_experiment(
        exp_id=cfg[0], model_name=cfg[1], spec_type=cfg[2], strategy=cfg[3],
        p1_lr=cfg[4], p1_ep=cfg[5], p2_lr=cfg[6], p2_ep=cfg[7]
    )
    all_results.append(r)
    df_cur = pd.DataFrame(all_results)
    df_cur.to_csv(res_csv_path, index=False)
    if DRIVE_DIR.exists():
        (DRIVE_DIR / 'results_groupv2').mkdir(parents=True, exist_ok=True)
        df_cur.to_csv(drive_csv_path, index=False)
        print(f'--> [ĐÃ LƯU VÀO GOOGLE DRIVE] Bảng cnn_benchmark_groupv2.csv đã cập nhật EXP-{eid:02d}!')

print('\n' + '='*80)
print('HOÀN TẤT TOÀN BỘ 13 THỰC NGHIỆM PROTOCOL B!')
print('='*80)
df_final = pd.DataFrame(all_results)
display_cols = ['exp_id', 'model_name', 'spec_type', 'strategy', 'val_accuracy', 'val_macro_f1', 'test_accuracy', 'macro_f1', 'training_time_min']
display(df_final[display_cols])


# GIAI ĐOẠN 6 (PHASE 6): MULTI-MODAL LATE FUSION TRÊN PROTOCOL B
Trong Protocol B, để loại bỏ hoàn toàn hiện tượng tối ưu trọng số trên tập test (Data Snooping):
1. **Tối ưu trọng số trên tập Validation:** Quét trọng số kết hợp `w` trên 360 mẫu Validation độc lập để tìm `best_w`.
2. **Đánh giá khách quan trên tập Test:** Cố định `best_w` và tính toán hiệu năng thực tế trên 720 mẫu Test hoàn toàn chưa từng biết trước.


In [ ]:
# 8. THỰC HIỆN PHASE 6: LATE FUSION VỚI CHỌN TRỌNG SỐ TRÊN VALIDATION SET
import joblib

split_file = Path('spectrograms/split_group_v2.csv')
if not split_file.exists(): split_file = Path('split_group_v2.csv')
if not split_file.exists(): split_file = DRIVE_DIR / 'split_group_v2.csv'
res_csv_path = Path('results_groupv2/cnn_benchmark_groupv2.csv')

if split_file.exists() and res_csv_path.exists():
    df_meta = pd.read_csv(split_file)
    val_meta = df_meta[df_meta['split'] == 'val'].reset_index(drop=True)
    test_meta = df_meta[df_meta['split'] == 'test'].reset_index(drop=True)
    
    y_val = np.array([GENRES.index(g) for g in val_meta['label']])
    y_test = np.array([GENRES.index(g) for g in test_meta['label']])
    
    df_bench = pd.read_csv(res_csv_path)
    best_row = df_bench.loc[df_bench['val_macro_f1'].idxmax()]
    best_eid = int(best_row['exp_id'])
    print(f"Mô hình CNN quán quân được chọn từ Validation: EXP-{best_eid:02d} ({best_row['model_name']})")
    
    cnn_val_probs = np.load(f'models/cnn_groupv2/exp{best_eid:02d}_val_probs.npy')
    cnn_test_probs = np.load(f'models/cnn_groupv2/exp{best_eid:02d}_test_probs.npy')
    
    # Nạp mô hình Fuzzy RF Protocol B (hoặc mô hình raw feature tốt nhất)
    rf_path = Path('models/best_random_forest_continuous_groupv2.joblib')
    if not rf_path.exists() and (DRIVE_DIR / 'models/best_random_forest_continuous_groupv2.joblib').exists():
        shutil.copy2(DRIVE_DIR / 'models/best_random_forest_continuous_groupv2.joblib', rf_path)
        
    if rf_path.exists():
        rf_model = joblib.load(rf_path)
        df_cont = pd.read_csv('data/datasets/features_fuzzy_continuous.csv') if Path('data/datasets/features_fuzzy_continuous.csv').exists() else pd.read_csv('features_fuzzy_continuous.csv')
        
        val_cont = df_cont[df_cont['sample_idx'].isin(val_meta['sample_idx'])].sort_values(by='sample_idx').reset_index(drop=True)
        test_cont = df_cont[df_cont['sample_idx'].isin(test_meta['sample_idx'])].sort_values(by='sample_idx').reset_index(drop=True)
        
        feat_cols = [c for c in df_cont.columns if c not in ['sample_idx', 'label', 'audio_path', 'segment_id']]
        
        rf_val_probs_raw = rf_model.predict_proba(val_cont[feat_cols].values)
        rf_test_probs_raw = rf_model.predict_proba(test_cont[feat_cols].values)
        
        # Align classes
        rf_classes = list(rf_model.classes_)
        rf_val_probs = np.zeros_like(rf_val_probs_raw)
        rf_test_probs = np.zeros_like(rf_test_probs_raw)
        for j, g in enumerate(GENRES):
            idx = rf_classes.index(g)
            rf_val_probs[:, j] = rf_val_probs_raw[:, idx]
            rf_test_probs[:, j] = rf_test_probs_raw[:, idx]
            
        # 1. Tìm trọng số tối ưu trên VALIDATION SET
        best_w, best_val_f1 = 0.5, 0.0
        val_records = []
        for w in np.linspace(0.0, 1.0, 21):
            w = round(w, 2)
            p_ens = w * cnn_val_probs + (1.0 - w) * rf_val_probs
            preds = p_ens.argmax(axis=1)
            f1 = f1_score(y_val, preds, average='macro') * 100
            val_records.append({'w_cnn': w, 'w_fuzzy': round(1.0 - w, 2), 'val_f1': round(f1, 2)})
            if f1 > best_val_f1:
                best_val_f1 = f1
                best_w = w
        
        print(f"Trọng số tối ưu tìm được trên Validation Set: w_cnn = {best_w} (Val F1 = {best_val_f1:.2f}%)")
        
        # 2. Đánh giá khách quan trên TEST SET với trọng số đã chọn
        final_test_p = best_w * cnn_test_probs + (1.0 - best_w) * rf_test_probs
        final_preds = final_test_p.argmax(axis=1)
        final_test_acc = accuracy_score(y_test, final_preds) * 100
        final_test_f1 = f1_score(y_test, final_preds, average='macro') * 100
        
        print("\n" + "="*70)
        print(f"KẾT QUẢ CUỐI CÙNG TRÊN TEST SET (PROTOCOL B - KHÔNG RÒ RỈ):")
        print(f"Độ chính xác Test Acc = {final_test_acc:.2f}%")
        print(f"Macro F1 Score       = {final_test_f1:.2f}%")
        print("="*70)
        
        df_fusion = pd.DataFrame([{
            'selected_w_cnn': best_w,
            'selected_w_fuzzy': round(1.0 - best_w, 2),
            'val_macro_f1': best_val_f1,
            'test_accuracy': round(final_test_acc, 2),
            'test_macro_f1': round(final_test_f1, 2)
        }])
        df_fusion.to_csv('results_groupv2/late_fusion_groupv2.csv', index=False)
        if DRIVE_DIR.exists():
            df_fusion.to_csv(DRIVE_DIR / 'results_groupv2/late_fusion_groupv2.csv', index=False)
    else:
        print('[THÔNG BÁO] Chưa tìm thấy mô hình Fuzzy RF Protocol B. Sau khi chạy notebook 03, hãy chạy lại ô này để late fusion.')


# ĐÓNG GÓI & TẢI KẾT QUẢ VỀ MÁY TÍNH
Chạy ô dưới để nén toàn bộ kết quả (`cnn_benchmark.csv`, các ma trận nhầm lẫn `.png`, các file xác suất `.npy`, checkpoint `.pth`) thành một file `results_colab.zip` và tự động tải về máy tính của bạn.


In [ ]:
# 9. TẢI FILE KẾT QUẢ PROTOCOL B VỀ MÁY
import shutil
from google.colab import files

print("Đang nén toàn bộ thư mục results_groupv2 và models/cnn_groupv2...")
shutil.make_archive('results_cnn_groupv2', 'zip', 'results_groupv2')
print("[OK] Đã tạo file results_cnn_groupv2.zip!")
files.download('results_cnn_groupv2.zip')
